# Spatiotemporal feature-engineering shotgun — XGBoost

This notebook screens **all registered feature families** against your existing selected-binary XGBoost baseline. The baseline already contains historical comparable-sale features. Every feature variant is fitted on the same training sample and predicted on the same unseen held-out homes.

**Default experiment:** all `SCREEN_FEATURE_VARIANTS` (including the six new advanced families), 10,000 training sales, up to 1,000 August 2025 validation sales, and 220 XGBoost trees. Residual corrections are **disabled by default**, since we already evaluated those separately. Switch `RUN_RESIDUALS=True` in the settings cell only if you want to rerun the older residual family experiments.

Results are exploratory: these sampled, reduced-tree scores should **not** match the full-history 700-tree XGBoost (7.82% MdAPE). The validation month must be before December 2025; reserve later months for confirmation, since December 2025–May 2026 has already been used repeatedly.

**Files:** `summary.csv`, `predictions.csv`, `failures.json` in the unique `OUTPUT_DIR` below. The experiment does not modify saved baseline models or the production prediction pipeline.


In [ ]:
import pandas as pd
from IPython.display import display
from spacetime_shotgun import run_shotgun, SCREEN_FEATURE_VARIANTS
from models.spacetime_screen_residuals import RESIDUAL_VARIANTS

VALIDATION_MONTH = "2025-08"
TRAIN_START = "2022-01-01"
TRAIN_CAP = 10000             # sample of historical completed sales
VALIDATION_CAP = 1000         # holdout sample, excluded from all training
SCREEN_TREES = 220           # small screen; full model uses 700
SEED = 42
OUTPUT_DIR = "outputs/spacetime_features_all"

RUN_RESIDUALS = False        # residual corrections tested separately
CALIBRATION_CAP = 650        # used only when RUN_RESIDUALS is True
CALIBRATION_FOLDS = 2
RESIDUAL_STRENGTH = 0.5

FEATURES = list(SCREEN_FEATURE_VARIANTS)
RESIDUALS = list(RESIDUAL_VARIANTS) if RUN_RESIDUALS else []

print(f"Feature candidates ({len(FEATURES)}):", FEATURES)
print("Residual candidates:", RESIDUALS)
print("Output:", OUTPUT_DIR)


In [ ]:
scores, predictions, failures = run_shotgun(
    validation_month=VALIDATION_MONTH,
    train_start=TRAIN_START,
    train_cap=TRAIN_CAP,
    validation_cap=VALIDATION_CAP,
    calibration_cap=CALIBRATION_CAP,
    calibration_folds=CALIBRATION_FOLDS,
    screen_trees=SCREEN_TREES,
    seed=SEED,
    features=FEATURES,
    residuals=RESIDUALS,
    residual_strength=RESIDUAL_STRENGTH,
    output_dir=OUTPUT_DIR,
)
display(scores)
if failures:
    print(f"{len(failures)} failed candidates; see {OUTPUT_DIR}/failures.json")
    display(pd.DataFrame(failures)[["arm", "variant", "error"]])
else:
    print("All requested candidates finished successfully.")


In [ ]:
# Compare feature candidates to the same reduced-sample XGBoost baseline.
display(scores.sort_values(["arm", "mdape"]))
display(scores.groupby("arm")[["mdape", "mape", "delta_mdape_pp", "delta_mape_pp"]]
        .agg(["min", "median"]).round(3))

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, max(5, 0.4 * len(scores) + 1)))
p = scores.sort_values("delta_mdape_pp")
ax.barh(p["arm"] + " / " + p["variant"], p["delta_mdape_pp"])
ax.axvline(0, linewidth=1, color="gray")
ax.set_xlabel("MdAPE change from baseline (percentage points; lower is better)")
ax.set_title("Spatiotemporal feature shotgun")
fig.tight_layout()
plt.show()


In [ ]:
# Inspect whether variants help particular property types.
import data
sales = data.load("listings")
sold_month = pd.to_datetime(sales["date_sold"]).dt.to_period("M")
names = sales.loc[sold_month == pd.Period(VALIDATION_MONTH),
                  ["ml_num", "house_type_name"]]
annotated = predictions.merge(names, on="ml_num", how="left", validate="many_to_one")
subgroups = (annotated.groupby(["arm", "variant", "house_type_name"])
             .agg(n=("ml_num", "size"), mdape=("abs_pct_error", "median"))
             .query("n >= 10")
             .sort_values(["house_type_name", "mdape"]))
display(subgroups.head(50))
print(f"Full result CSV: {OUTPUT_DIR}/predictions.csv")


## Next: refine the winners

Review `summary.csv` and `failures.json`. Keep 3–5 candidates with the strongest MdAPE/MAPE and reasonable runtime. Confirm them across multiple **earlier** validation months in `spacetime_three_month.ipynb` using 40,000 training examples and 500 trees; for that notebook set `FEATURE_VARIANTS` to the shortlist and `RESIDUAL_VARIANTS=[]`.

A small change on one month can be noise or selection bias. Only promote persistent improvements to full-history 700-tree validation. The baseline and shortlisted variants must use identical chronological splits and target definitions.

If your local repo is behind, update it and open this notebook in Jupyter:

```powershell
git switch main
git pull origin main
uv sync
uv run jupyter lab
```

For a quicker initial screen, change `TRAIN_CAP=5000`, `VALIDATION_CAP=400`, and `SCREEN_TREES=100`; for a stronger screen increase those settings, but retain the same holdout across variants.
